# Extracting member names and task descriptions from raw attributions with LLMs

In this notebook, different open-source LLMs (Mixtral, LLaMA, Gemma) are used to extract member names along with the sentences describing the tasks they performed within a team, based on raw attribution texts.

The first section loads the teams’ roster data and their full raw attribution texts, collected in `6_match_teams_2022_attributions.ipynb`. In the second section, variables for LLM model configuration are defined. Ollama is the framework used for running the models. This section also includes functions for building two types of prompts.

The third section contains functions for quickly testing different prompts, models, and prompting techniques by manually comparing outputs with 5 validation teams. Once the final prompt is written with clear rules and examples of edge cases, it is used in the fourth section to evaluate the models. Each of the 14 evaluation teams (see `"8_manually_annotate_team_attributions"`) is processed twice: first by roster batches, then name by name.

The results are saved in `"../data/attributions/2022_attributions/llm_json_outputs/"` and used for model evaluation in the `10_llm_extraction_evaluation.ipynb` notebook.


In [1]:
import requests
import json
import pandas as pd
from collections import defaultdict, Counter
from time import sleep
import re

## 1. Data

- roster data
- raw attributions texts

In [2]:
df_roster = pd.read_table("../data/attributions/2022_attributions/team_rosters_2022.tsv")

In [3]:
df_roster_teams_people = df_roster[["Team", "TeamID", "FullName"]]
df_roster_teams_people

,Team,TeamID,FullName
0,AFCM-Egypt,4140,Moetaz Sherif Mohamed Radwan Metawea
1,AFCM-Egypt,4140,Omar Ahmed Abdalla
2,AFCM-Egypt,4140,Ahmad Mahmoud Galal
3,AFCM-Egypt,4140,Ahmed Elshazly
4,AFCM-Egypt,4140,Mahmoud Mohammed AbdelGawad
...,...,...,...
7757,iBowu-China,4236,Yanbo Han
7758,iBowu-China,4236,Yifei Chen
7759,iBowu-China,4236,Yufei Zhang
7760,iBowu-China,4236,Yuhao Jin


In [4]:
team_meta_df = pd.read_table(
    "../data/igem_scrapping_2025/team_meta_full.tsv",
    usecols=["Team", "TeamID"]
)
team_meta_df = team_meta_df.rename(columns={"Team": "TeamMetaName"})

team_meta_df

,TeamID,TeamMetaName
0,5260,ABOA
1,5794,ABOA
2,4831,ABOA-Turku
3,4068,ABSI_Kenya
4,2815,ACIBADEM_ISTANBUL
...,...,...
5077,3271,uOttawa
5078,5293,uOttawa
5079,807,uOttawa_CA
5080,5357,ucl


We will check if team names are the same in the roster and in the team meta data.

In [5]:
# Add a TeamMetaName column to check if there are cases when the team name found in roster data is a bit different than the one in the team_meta_full (different dashes used)
# Left merge on TeamID the roster teams with the team meta to get the TeamMetaName
df_roster_meta = df_roster_teams_people.merge(
    team_meta_df[["TeamID", "TeamMetaName"]],
    on="TeamID",
    how="left"  
)

df_roster_meta = df_roster_meta.rename(columns={"Team": "TeamRosterName"})

df_roster_meta

,TeamRosterName,TeamID,FullName,TeamMetaName
0,AFCM-Egypt,4140,Moetaz Sherif Mohamed Radwan Metawea,AFCM-Egypt
1,AFCM-Egypt,4140,Omar Ahmed Abdalla,AFCM-Egypt
2,AFCM-Egypt,4140,Ahmad Mahmoud Galal,AFCM-Egypt
3,AFCM-Egypt,4140,Ahmed Elshazly,AFCM-Egypt
4,AFCM-Egypt,4140,Mahmoud Mohammed AbdelGawad,AFCM-Egypt
...,...,...,...,...
7757,iBowu-China,4236,Yanbo Han,iBowu-China
7758,iBowu-China,4236,Yifei Chen,iBowu-China
7759,iBowu-China,4236,Yufei Zhang,iBowu-China
7760,iBowu-China,4236,Yuhao Jin,iBowu-China


In [6]:
df_roster_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7762 entries, 0 to 7761
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   TeamRosterName  7762 non-null   object
 1   TeamID          7762 non-null   int64 
 2   FullName        7762 non-null   object
 3   TeamMetaName    7762 non-null   object
dtypes: int64(1), object(3)
memory usage: 242.7+ KB


In [7]:
df_roster_meta[df_roster_meta["TeamMetaName"] != df_roster_meta["TeamRosterName"]]

,TeamRosterName,TeamID,FullName,TeamMetaName


There are no such rows where roster and meta team names are different so we can just use the "Team" column from the first roster df when looking up the roster lists of specific teams.

In [8]:
# Save Teams and FullNames in a dictionary (to be used for LLM prompts)
roster_2022_dict = (
    df_roster
    .groupby("Team")["FullName"]
    .apply(list)
    .to_dict()
)

In [9]:
# Helper functions for getting team attribution text and roster

# Load the JSON file containing all teams' attributions data
def load_attributions_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Return team's full attribution text from raw attributions data
def get_team_attributions_text(data, team_name):
    for entry in data:
        if entry["teamName"] == team_name:
            return entry["attributionsText"]
    return None

# Return the list of member names for given team from the roster dictionary
def get_team_roster(roster_dict, team_name):
    return roster_dict.get(team_name, [])

In [ ]:
# raw_attributions_data = load_attributions_json("../data/attributions/2022_attributions/raw_attributions_2022.json")
raw_attributions_data = load_attributions_json("../data/attributions/2022_attributions/html_attributions_2022.json")
raw_validation_attributions_data = load_attributions_json("../data/attributions/2022_attributions/manual_annotation/5_validation_teams_raw_attributions.json")

## 2.LLM Config and Build Prompt

Ollama is the framework used for running and managing large language models locally with CLI commands. A few open-source models pulled from Ollama are tried out here: mixtral:8x7b, llama3.3:70b, and gemma3:27b. Custom variants of these models (mixtral-json, llama-json, and gemma-json—were) were created locally using Ollama’s create command. These custom models were wrapped with simple system-level instructions like: "You are an AI that extracts structured JSON from text according to instructions. Always output only valid JSON."

The LLaMA model is the largest and slowest, with 70 billion parameters, but it produced the best results on the 5 validation teams. The code below is run on GPU, on Mac Studio with 256Gb RAM.

In [11]:
# LLM config

OLLAMA_API_URL = "http://localhost:11434/api/generate"
# MODEL_NAME = "mixtral-json"   # works worse than llama
MODEL_NAME = "llama-json"  #too slow
# MODEL_NAME = "gemma-json"  # smallest model, similar to mixtral
NUM_RUNS_PER_PROMPT = 3      # number of times to query the LLM per member for majority vote, ADD LATER: increase the number
MAX_ATTEMPTS = 3 # reprompt max 2 more times if invalid json is given in the response
BATCH_SIZE = 4

Two functions are defined for building prompts:

- One uses the entire roster (or batches of it) in a single prompt, asking the model to identify all listed names along with their corresponding task descriptions.
- The other prompts for one name at a time, asking the model to find the specified member and extract only their task descriptions.

Multiple versions of the prompt were tried out, but the following ones produced the best initial results on the validation teams. One version involved not providing the model with any roster names, instead allowing it to identify names and tasks independently, with relevant names later fuzzy matched with the roster.  However, with this prompt version, the model would omit a lot of roster names, so it was discarded.

In [12]:
# Build LLM prompt using team members list (full or batches of names) and team raw attributions text

def build_prompt(member_names, team_raw_attributions):
    formatted_names = "\n".join(f"- {name}" for name in member_names)
    example = '''

Here is an example (for illustration only, do NOT use these names in your answer)


Example Roster:
- Alice Johnson
- B�b Smith
- Charlie Brown


Example Raw Attribution Text:
"iGEM Project Description Team Members Alicé Johnson Team Leader Wet Lab Dry Lab Finances Alicé was responsible for labs, especially research, design of the SynBio solution, and modeling. She also helped with the team's finances and managed communication with various sponsors.\n Bob Smith Communications Bob worked in Communications, writing and producing the promotion video, and networking with other iGEM teams. All team members were a part of the project brainstorming."


Example Output:
[
 {
   "RosterName": "Alice Johnson",
   "RawTextName": "Alicé Johnson",
   "TasksDescription": "Wet Lab Dry Lab Finances Alicé was responsible for labs, especially research, design of the SynBio solution, and modeling. She also helped with the team's finances and managed communication with various sponsors. All team members were a part of the project brainstorming."
 },
 {
   "RosterName": "B�b Smith",
   "RawTextName": "Bob Smith",
   "TasksDescription": "Communications Bob worked in Communications, writing and producing the promotion video, and networking with other iGEM teams. All team members were a part of the project brainstorming."
 },
 {
   "RosterName": "Charlie Brown",
   "RawTextName": null,
   "TasksDescription": null
 }
]
'''

    return f"""
You are an AI that extracts structured JSON from text according to the following rules.

Your Task:
From the given raw attributions text, process the chunk of the official team roster list and return a JSON array where:
- Each object corresponds to one name from the official team roster list
- Fields:
    - RosterName: Exact name from the official roster list (always included for every name in the roster)
    - RawTextName: Approximately matched name found in the raw text (leave null if a similar name is not mentioned)
    - TasksDescription: Extracted phrases or sentences that describe tasks the person performed in the team (leave null if none are mentioned).

Rules:
- Each RosterName must appear only once in the JSON array. If multiple references to the same person appear later on in the raw text, merge all their task descriptions into the same TasksDescription string.
- Match names from the official roster with names in the raw text using approximate matching, accounting for spelling variations and cases where only the first or last name appears in the text
- Do not add names that are not in the official team roster list (ignore extra names in the raw text)
- If a sentence applies to multiple people (e.g., “all team members did wiki coding”), include that sentence for all relevant names
- Include only real text from the raw text input (do not make up tasks) and do not change the spelling of roster or raw text names
- Ignore roles or titles, focus on the person's task contributions
- If RawTextName is found but no tasks are described, leave TasksDescription null

{example}

Now process the actual input bellow:

Official Team Roster:
{formatted_names}

Raw Attribution Text:
\"\"\"{team_raw_attributions}\"\"\"

Expected Output Format:
[
 {{"RosterName": "<from official list>", "RawTextName": "<from text or null>", "TasksDescription": "<from text or null>"}},
 ...
]

Return ONLY a valid JSON array, no code, no explanations.

"""

In [ ]:
# Build LLM prompt for a ONE roster name at a time and raw attribution text
# It gives better results and less hallucinations than the previous version, but takes more time as a separate prompt needs to be sent for each member

def build_prompt_single(roster_name, team_raw_attributions):
    example = '''
Example RosterName: B�b Smith

Example Raw Attribution Text:

"iGEM Project Description Team Members Alicé Johnson Team Leader Wet Lab Dry Lab Finances Alicé was responsible for labs, especially research, design of the SynBio solution, and modeling. She also helped with the team's finances and managed communication with various sponsors.\n Bob Smith Communications Bob worked in Communications, writing and producing the promotion video, and networking with other iGEM teams. All team members were a part of the project brainstorming."

Example Output:
{
  "RosterName": "B�b Smith",
  "RawTextName": "Bob Smith",
  "TasksDescription": "Communications Bob worked in Communications, writing and producing the promotion video, and networking with other iGEM teams. All team members were a part of the project brainstorming."
}
'''

    return f"""
You are an AI that extracts structured JSON from text according to the following rules.

Your Task:
From the given raw attributions text, find all references to the given person and return a single JSON object with:
- RosterName: The exact roster name provided below (always included)
- RawTextName: The matched name from the raw text (or null if not mentioned)
- TasksDescription: Extracted phrases or sentences from the text that describe tasks that this person did in the team (or null if none found)

Rules:
- Match name approximately (account for spelling differences and cases where only first or last name is mentioned)
- Do NOT invent tasks, use only what is present in the text
- If multiple references to the person exist, merge all task descriptions into a single string
- If a sentence in the raw text applies to multiple people (e.g., “all team members did wiki coding”), include that sentence for the person (if it relates to them as well)
- Ignore roles or titles, focus on the person's task contributions
- If the person is not mentioned, keep RawTextName and TasksDescription as null

RosterName to look for:
{roster_name}

Raw Attribution Text:
\"\"\"{team_raw_attributions}\"\"\"

Example Input and Output:
{example}

Expected Output Format:
{{
  "RosterName": "<roster name>",
  "RawTextName": "<matched name from text or null>",
  "TasksDescription": "<relevant phrases/sentences or null>"
}}

Return only a valid JSON object, nothing else.
"""

In [ ]:
# Build LLM prompt for a ONE roster name at a time and raw HTML attributions (with username and name)

def build_html_prompt_single(roster_name, username, team_raw_attributions):
    example = '''
Example RosterName (Primary): Bb Smith
Example Username (Secondary): Bsmith123

Example HTML Markup:

"
<html lang="en">
  <head>
    <meta charset="utf-8" />
    <title>Blah Blah</title>
  </head>
  <body>
    <section>
      <pre>"iGEM Project Description Team Members Alicé Johnson Team Leader Wet Lab Dry Lab Finances Alicé was responsible for labs, especially research, design of the SynBio solution, and modeling. She also helped with the team's finances and managed communication with various sponsors.\n Bob Smith Communications Bob worked in Communications, writing and producing the promotion video, and networking with other iGEM teams. All team members were a part of the project brainstorming."</pre>
    </section>
  </body>
</html>
"

Example Output:
{
  "RosterName": "B�b Smith",
  "RawTextName": "Bob Smith",
  "TasksDescription": "Communications Bob worked in Communications, writing and producing the promotion video, and networking with other iGEM teams. All team members were a part of the project brainstorming."
}
'''

    return f"""
You are an AI that extracts structured JSON from HTML markup according to the following rules.

Your Task:
From the given raw HTML markup with attributions, find all references to the given person and return a single JSON object with:
- RosterName: The exact roster name provided below (always included)
- RawTextName: The matched name from the HTML markup (or null if not mentioned)
- TasksDescription: Extracted TEXT (phrases or sentences) from the HTML markup that describe tasks that this person did in the team (or null if none found)

Rules:
- Match name approximately (account for spelling differences and cases where only first or last name is mentioned).
- Prioritize matching the person using the RosterName.
- If the RosterName is not found, attempt to match using the Username as an alternative name mentioned in the raw text.
- Do NOT invent tasks, use only what is present in the HTML markup
- If multiple references to the person exist, merge all task descriptions into a single string
- If a sentence in the HTML markup applies to multiple people (e.g., “all team members did wiki coding”), include that sentence for the person (if it relates to them as well)
- Ignore roles or titles, focus on the person's task contributions
- If the person is not mentioned by either the RosterName or the Username, keep RawTextName and TasksDescription as null

RosterName (Primary Identifier) to look for:
{roster_name}

Username (Secondary Identifier) to look for:
{username}

Raw HTML Markup:
\"\"\"{team_raw_attributions}\"\"\"

Example Input and Output:
{example}

Expected Output Format:
{{
  "RosterName": "<roster name>",
  "RawTextName": "<matched name from text or null>",
  "TasksDescription": "<relevant phrases/sentences or null>"
}}

Return only a valid JSON object, nothing else.
"""

## 3. Validate for One Team

This section is part of the prompt engineering process. Different prompt versions and prompting strategies are tested on the 5 validation teams by manually comparing their results. The best prompt version and the name-by-name prompting strategy are selected for the final model. The functions defined here are therefore used only for quick testing and are later refined in the fourth section of the notebook.

In [ ]:
# Get attributions text and team members for one example team

test_team_name = "vilnius-Lithuania"

test_attributions_text = get_team_attributions_text(raw_attributions_data, test_team_name)

if test_attributions_text is None:
    print(f"No attributions text found for team: {test_team_name}")

test_member_names = get_team_roster(roster_2022_dict, test_team_name)

### 3.1 Prompt for Full Roster

In [24]:
test_prompt = build_prompt(test_member_names, test_attributions_text)

In [ ]:
# For full team roster prompt

# Send request to Ollama local API
response = requests.post(
    OLLAMA_API_URL,
    json={
        "model": MODEL_NAME,  # custom model I created in CLI
        "prompt": test_prompt,
        "temperature": 0.0,
        "stream": False #full response, not stream
    }
)

data = response.json()
raw_output = data.get("response", "").strip()

# Remove markdown code block markers - important for LLAMA 
cleaned_output = re.sub(r"^```[a-z]*\n|\n```$", "", raw_output.strip())

try:
    parsed = json.loads(cleaned_output)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("Invalid JSON after cleaning. Raw Output:\n", raw_output)


### 3.2 Prompt Name per Name

In [25]:
# For single member prompts

test_single_prompts = []
for name in test_member_names:
    test_single_prompt = build_prompt_single(name, test_attributions_text)
    test_single_prompts.append(test_single_prompt)

In [ ]:
# For single member prompts

results = []

for test_single_prompt in test_single_prompts:
    payload = {
        "model": MODEL_NAME,
        "prompt": test_single_prompt,
        "temperature": 0.0,
        "stream": False
    }

    attempt = 0
    max_attempts = 3 # reprompt max 2 more times if invalid json is given in the response
    parsed = None

    while attempt < max_attempts and parsed is None:
        try:
            response = requests.post(OLLAMA_API_URL, json=payload, timeout=120)
            if response.status_code != 200:
                print(f"Error {response.status_code}: {response.text}")
                break

            data = response.json()
            raw_output = data.get("response", "").strip()

            cleaned_output = re.sub(r"^```[a-z]*\n|\n```$", "", raw_output.strip(), flags=re.MULTILINE)

            try:
                parsed = json.loads(cleaned_output)
                results.append(parsed)
                print(json.dumps(parsed, indent=2, ensure_ascii=False))  # Immediate print
            except json.JSONDecodeError:
                print(f"Invalid JSON attempt {attempt+1}. Retrying...")
                attempt += 1
                if attempt < max_attempts:
                    sleep(1)
        except requests.RequestException as e:
            print(f"Request failed: {e}")
            break

    if parsed is None:
        print(f"Failed after {max_attempts} attempts for this prompt:\n{test_single_prompt}")

    sleep(0.5)


### 3.3 Prompt for Roster Batches

The `chunk_list` function is used in the 4th section as well, so do not forget to run the following cell.

In [27]:
# Helper function to chunk the roster names into batches 
def chunk_list(roster_list, size):
    for i in range(0, len(roster_list), size):
        yield roster_list[i:i + size]

In [42]:
# Testing that chunking works

# for chunk in chunk_list(test_member_names, 4):
#     print(chunk)

In [88]:
# Process names in batches of 4 or whatever we set it as (one API call for each 4 names or leftovers)
        
# Main function to send requests in batches

def process_roster_in_batches_1(member_names, team_raw_attributions, batch_size):

    results = []

    for chunk in chunk_list(member_names, batch_size):
        prompt = build_prompt(chunk, team_raw_attributions)

        payload = {
            "model": MODEL_NAME,
            "prompt": prompt,
            "temperature": 0.0,
            "stream": False
        }

        attempt = 0
        parsed = None

        while attempt < MAX_ATTEMPTS and parsed is None:
            try:
                response = requests.post(OLLAMA_API_URL, json=payload, timeout=120)
                if response.status_code != 200:
                    print(f"Error {response.status_code}: {response.text}")
                    break

                data = response.json()
                raw_output = data.get("response", "").strip()

                cleaned_output = re.sub(r"^```[a-z]*\n|\n```$", "", raw_output.strip())

                try:
                    parsed = json.loads(cleaned_output)
                    if isinstance(parsed, list):
                        results.extend(parsed)
                except json.JSONDecodeError:
                    print(f"Invalid JSON attempt {attempt+1}. Retrying...")
                    attempt += 1
                    if attempt < MAX_ATTEMPTS:
                        sleep(1)

            except requests.RequestException as e:
                print(f"Request failed: {e}")

        if parsed is None:
            print(f"Failed after {MAX_ATTEMPTS} attempts for this prompt:\n{prompt}")

            sleep(0.5) 

    return results

In [ ]:
all_results = process_roster_in_batches_1(test_member_names, test_attributions_text, 2)

print(json.dumps(all_results, indent=2, ensure_ascii=False))

## 4. Teams for Model Evaluation

We will evaluate the model’s performance by comparing its outputs to the ground truth: 14 manually annotated teams with member names and their corresponding task descriptions. Since the model is not fully deterministic and can produce different outputs even with the temperature set to 0, we will prompt the model multiple times for the same teams and save the outputs from all rounds to obtain a more reliable accuracy estimate. Prompting is done in this section, while the evaluation is done in the notebook `10_llm_extraction_evaluation.ipynb`.

The prompting is first done using roster batches (4 or less names per prompt), and then with one member name at the time. The first approach is faster since it requires fewer prompts, but it tends to yield worse results: the model may hallucinate and miss some names, or miss sentences that describe tasks for all team members. Prompting with one name at a time improves focus and leads to more thorough extraction of task descriptions.

Results are saved in `"../data/attributions/2022_attributions/llm_json_outputs/"`.

In [14]:
tasks_evaluation_teams = load_attributions_json("../data/attributions/2022_attributions/manual_annotation/14_test_teams.json")

In [101]:
evaluation_teams = [list(team_dict.keys())[0] for team_dict in tasks_evaluation_teams]

In [ ]:
# teams that do not work well with batches - prompting with one name at a time works better and faster
# evaluation_teams = ['TU_Braunschweig',
#  'Technion-Israel']

In [16]:
# Clean JSON output of markdown ```
def clean_output(raw_output):
    return re.sub(r"^```[a-z]*\n|\n```$", "", raw_output.strip(), flags=re.MULTILINE)

### 4.1 Roster Batches

In [35]:
# Query Ollama with retry logic for invalid JSON
def query_model_with_retries(prompt):
    attempt = 0
    parsed = None

    while attempt < MAX_ATTEMPTS and parsed is None:
        try:
            response = requests.post(
                OLLAMA_API_URL,
                json={"model": MODEL_NAME, "prompt": prompt, "temperature": 0.0, "stream": False},
                timeout=240
            )

            if response.status_code != 200:
                print(f"Error {response.status_code}: {response.text}")
                attempt += 1
                sleep(1)
                continue

            data = response.json()
            raw_output = data.get("response", "").strip()
            cleaned_output = clean_output(raw_output)

            try:
                parsed = json.loads(cleaned_output)
                if not isinstance(parsed, list):
                    parsed = None
                    raise ValueError("Output is not a list")
            except json.JSONDecodeError:
                print(f"Invalid JSON attempt {attempt + 1}. Retrying...")
                attempt += 1
                sleep(1)

        except requests.RequestException as e:
            print(f"Request failed: {e}")
            attempt += 1
            sleep(2)

    return parsed

In [36]:
# Main batch processing
def process_roster_in_batches(member_names, team_raw_attributions, batch_size):
    # Prepare container for multiple rounds
    rounds_data = {f"Round{r+1}": [] for r in range(NUM_RUNS_PER_PROMPT)}

    for chunk in chunk_list(member_names, batch_size):
        print(f"Processing chunk: {chunk}")

        prompt = build_prompt(chunk, team_raw_attributions)

        # For each round (NUM_RUNS_PER_PROMPT)
        for run_index in range(NUM_RUNS_PER_PROMPT):
            parsed = query_model_with_retries(prompt)
            if parsed:
                # Append entire parsed list for the chunk to its round
                rounds_data[f"Round{run_index+1}"].extend(parsed)

        sleep(0.5)

    return rounds_data


In [ ]:
# Main processing loop for all test teams

results = []

for team_name in evaluation_teams:
    print(f"Processing team: {team_name}")
    team_raw_attributions = get_team_attributions_text(raw_attributions_data, team_name)
    if not team_raw_attributions:
        print(f"No attributions text for team: {team_name}")
        continue

    team_members = get_team_roster(roster_2022_dict, team_name)
    team_rounds = process_roster_in_batches(team_members, team_raw_attributions, BATCH_SIZE)

    results.append({team_name: team_rounds})

    with open("../data/attributions/2022_attributions/llm_json_outputs/14_teams_model_outputs_batches.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)


In [70]:
len(results)

12

In [ ]:
with open("../data/attributions/2022_attributions/manual_annotation/14_teams_model_outputs_3_rounds.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

### 4.2 Name per Name

In [ ]:
# Query Ollama with retry logic for invalid JSON
def query_model_single_name(prompt):
    attempt = 0
    parsed = None

    while attempt < MAX_ATTEMPTS and parsed is None:
        try:
            response = requests.post(
                OLLAMA_API_URL,
                json={"model": MODEL_NAME, "prompt": prompt, "temperature": 0.0, "stream": False},
                timeout=120
            )

            if response.status_code != 200:
                print(f"Error {response.status_code}: {response.text}")
                attempt += 1
                sleep(1)
                continue

            data = response.json()
            raw_output = data.get("response", "").strip()
            cleaned_output = clean_output(raw_output)

            try:
                parsed = json.loads(cleaned_output)
                if isinstance(parsed, dict):
                    parsed = [parsed]  
                elif not isinstance(parsed, list):
                    parsed = None
                    raise ValueError("Output is neither a list nor a dict")
            except json.JSONDecodeError:

                print(f"Invalid JSON attempt {attempt + 1}. Retrying...")
                attempt += 1
                sleep(1)

        except requests.RequestException as e:
            print(f"Request failed: {e}")
            attempt += 1
            sleep(2)

    return parsed

In [104]:
def process_roster_single(member_names, team_raw_attributions):
    # Prepare container for multiple rounds
    rounds_data = {f"Round{r+1}": [] for r in range(NUM_RUNS_PER_PROMPT)}

    for name in member_names:
        print(f"Processing member: {name}")

        # For each round (NUM_RUNS_PER_PROMPT)
        for run_index in range(NUM_RUNS_PER_PROMPT):
            prompt = build_prompt_single(name, team_raw_attributions)
            parsed = query_model_single_name(prompt)
            if parsed:
                # For single-name prompts, parsed might already be one dict or a list with one dict
                if isinstance(parsed, list):
                    rounds_data[f"Round{run_index+1}"].extend(parsed)
                else:
                    rounds_data[f"Round{run_index+1}"].append(parsed)

        sleep(0.5)

    return rounds_data

In [107]:
results = []

for team_name in evaluation_teams:
    print(f"Processing team: {team_name}")
    team_raw_attributions = get_team_attributions_text(raw_attributions_data, team_name)
    if not team_raw_attributions:
        print(f"No attributions text for team: {team_name}")
        continue

    team_members = get_team_roster(roster_2022_dict, team_name)
    team_rounds = process_roster_single(team_members, team_raw_attributions)

    results.append({team_name: team_rounds})

    with open("../data/attributions/2022_attributions/llm_json_outputs/14_teams_model_outputs_single_name_prompt.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

Processing team: Aalto-Helsinki
Processing member: Ilse Kaaja
Processing member: Juuso Taskinen
Processing member: Rishi Banerjee
Processing member: Sami Jalil
Processing member: A. Sesilja Aranko
Processing member: Ville Paavilainen
Processing member: Heli Viskari
Processing member: Markus Linder
Processing member: Melissa Hendr�n
Processing member: Veera Kurki
Processing member: Amna Gul
Processing member: Anniina Könönen
Processing member: Diogo Dias
Processing member: Hanna Nebelung
Processing member: Joose Lankia
Processing member: Lilith Heiland
Processing member: Mari Keskiivari
Processing member: Zsofia Hesketh
Processing team: Aboa
Processing member: Pauli Kallio
Processing member: Juuli Hietarinne
Processing member: Malin Eriksson
Processing member: Anni Marjomaa
Processing member: Evelina Ojaniittu
Processing member: Iida Raaska
Processing member: Jesper Mickos
Processing member: Jonna Pohjankukka
Processing member: Kevät Sova
Processing member: Kristiina Keski-Oja
Processin